## BOM

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("data")

# If the notebook is inside notebooks/
if not DATA_DIR.exists():
    DATA_DIR = Path("../data")

bom_raw = pd.read_csv(DATA_DIR / "bom_materials.csv")

bom_raw

### Data Cleaning

In [ ]:
# Data Cleaning
bom = bom_raw.dropna(subset=["material_family"]).copy()

bom["product"] = bom["product_ref"].map({
    "RM6_NE_IQI_433520": "RM6",
    "RMAirSeT IIQ 24kV IEC": "RM AirSeT"
})

bom = bom.reset_index(drop=True)

bom.head()

### Missing Value

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": bom.isna().sum(),
    "missing_pct": bom.isna().mean() * 100
})

missing_summary.round(2)

In [ ]:
missing_bom = bom.loc[
    bom["mass_kg"].isna() | bom["share_pct"].isna() | bom["eu_series_ids"].isna() | bom["ww_series_ids"].isna(),
    [
        "product",
        "material_label_source",
        "material_family",
        "share_pct",
        "mass_kg",
        "eu_series_ids",
        "ww_series_ids",
        "comment"
    ]
]

missing_bom

### Material Composition Analysis

In [ ]:
material_mass = bom.pivot_table(
    index="material_family",
    columns="product",
    values="mass_kg",
    aggfunc="sum",
    min_count=1
)

In [ ]:
material_mass = (
    bom.groupby(["material_family", "product"])["mass_kg"]
    .sum(min_count=1)
    .unstack()
)

material_mass

In [ ]:
material_mass.plot(
    kind="bar",
    figsize=(12, 5),
    width=0.75,
    color=["#3478B8", "#3DCD58"]
)

plt.title("Known Material Mass by Product")
plt.xlabel("Material Family")
plt.ylabel("Mass per Product (kg)")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Product")
plt.tight_layout()
plt.show()

### BOM Data Coverage Analysis

In [ ]:
rm_prices = pd.read_csv(
    DATA_DIR / "rm_prices_monthly_wide.csv"
)

available_series = set(
    rm_prices.drop(
        columns=["month", "year", "month_num"]
    )
    .dropna(axis=1, how="all")
    .columns
)

available_series

In [ ]:
def has_available_price(series_ids):
    if pd.isna(series_ids):
        return False

    ids = series_ids.split(";")

    return any(
        series_id in available_series
        for series_id in ids
    )


bom["has_price_proxy"] = (
    bom["eu_series_ids"].apply(has_available_price)
)

bom[[
    "product",
    "material_family",
    "share_pct",
    "eu_series_ids",
    "has_price_proxy"
]]

#### Calculate confirmed coverage

In [ ]:
confirmed_coverage = (
    bom.loc[bom["has_price_proxy"]]
    .groupby("product")["share_pct"]
    .sum()
)

confirmed_coverage

#### Build coverage summary

In [ ]:
coverage = pd.DataFrame(
    index=["RM6", "RM AirSeT"]
)

coverage["Confirmed & price proxy available"] = (
    confirmed_coverage
)

# Assumption from the original BOM annotation
coverage["Carbon steel to confirm"] = [0.45, 0.0]

coverage["Unknown / not priceable"] = (
    1 - coverage.sum(axis=1)
)

coverage_pct = coverage * 100

coverage_pct.round(1)

#### Visualize BOM coverage

In [ ]:
ax = coverage_pct.plot(
    kind="barh",
    stacked=True,
    figsize=(11, 4),
    color=["#3DCD58", "#E9B44C", "#BDBDBD"],
    width=0.5
)

for container in ax.containers:
    labels = [
        f"{bar.get_width():.1f}%"
        if bar.get_width() > 0 else ""
        for bar in container
    ]

    ax.bar_label(
        container,
        labels=labels,
        label_type="center",
        fontsize=11
    )

ax.set_title("BOM Mass Coverage by Product", fontsize=15)
ax.set_xlabel("Share of Total Product Mass (%)")
ax.set_ylabel("")
ax.set_xlim(0, 100)

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.2),
    ncol=3
)

plt.tight_layout()
plt.show()

### Price Mapping

#### Price Data Availability

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

file_path = "data/RM_Prices_XHEC_Challenge.xlsx"

catalog = pd.read_excel(
    file_path, sheet_name="Source_Catalog"
)

prices = pd.read_excel(
    file_path, sheet_name="Monthly_Wide"
)

bom = pd.read_csv("data/bom_materials.csv")

# Remove summary rows
bom = bom[bom["material_family"].notna()].copy()

catalog.head()

In [ ]:
catalog["proxy_type"].value_counts().plot(
    kind="bar",
    figsize=(9, 4),
    title="Price Series by Proxy Type",
    rot=45
)

plt.ylabel("Number of Series")
plt.tight_layout()
plt.show()

In [ ]:
series_ids = catalog["series_id"].tolist()

price_availability = pd.DataFrame({
    "series_id": series_ids,
    "n_available": prices[series_ids].notna().sum().values,
    "n_missing": prices[series_ids].isna().sum().values
})

price_availability["coverage"] = (
    price_availability["n_available"] / len(prices)
)

price_availability["has_data"] = (
    price_availability["n_available"] > 0
)

price_availability

In [ ]:
catalog = catalog.merge(
    price_availability,
    on="series_id",
    how="left",
    validate="one_to_one"
)

catalog[[
    "series_id",
    "material",
    "proxy_type",
    "access",
    "n_available",
    "coverage",
    "has_data"
]]

#### Series Mapping

In [ ]:
mapping = bom.melt(
    id_vars=[
        "product_ref",
        "material_label_source",
        "material_family",
        "share_pct",
        "mass_kg"
    ],
    value_vars=[
        "eu_series_ids",
        "ww_series_ids"
    ],
    var_name="mapping_region",
    value_name="series_id"
)

mapping["mapping_region"] = mapping["mapping_region"].map({
    "eu_series_ids": "EU",
    "ww_series_ids": "WW"
})

# Split multiple series IDs
mapping["series_id"] = mapping["series_id"].str.split(";")

# Expand each series ID into an individual row
mapping = mapping.explode("series_id")

mapping["series_id"] = mapping["series_id"].str.strip()

mapping = mapping.reset_index(drop=True)

mapping.head(10)

#### Mapping status

In [ ]:
mapping = mapping.merge(
    catalog,
    on="series_id",
    how="left",
    validate="many_to_one"
)

mapping.head()

In [ ]:
mapping_result = mapping[[
    "product_ref",
    "material_label_source",
    "material_family",
    "mapping_region",
    "series_id",
    "series_name",
    "unit",
    "currency",
    "proxy_type",
    "access",
    "coverage",
    "has_data"
]]

mapping_result

In [ ]:
mapping["mapping_status"] = np.select(
    [
        mapping["series_id"].isna(),
        mapping["series_name"].isna(),
        mapping["has_data"].eq(True)
    ],
    [
        "No mapping",
        "Unknown series",
        "Available"
    ],
    default="Mapped but unavailable"
)

mapping["mapping_status"].value_counts()

#### Mapping Summary

In [ ]:
material_summary = mapping.groupby(
    [
        "product_ref",
        "material_label_source",
        "material_family"
    ],
    as_index=False
).agg(
    n_mapped=("series_id", "count"),
    n_available=("has_data", lambda x: x.eq(True).sum())
)

material_summary["has_available_proxy"] = (
    material_summary["n_available"] > 0
)

material_summary

## Findings

- The estimated total product mass is approximately 310 kg for RM6 and 629 kg for RM AirSeT.
- Confirmed materials with available price proxies represent 40.7% of RM6's total mass and 73.1% of RM AirSeT's total mass.
- Approximately 45% of RM6's mass may correspond to carbon steel, pending client confirmation.
- Unidentified or non-priceable materials limit the completeness of the current cost model.
- Additional BOM information and baseline purchase prices are required to improve cost estimation accuracy.